# Début Rapport d'analyse

problème avec ces colonnes : 'ip_s', 'ip_d', 'sport', 'dport', 'flags', 'n_pkt_src', 'n_pkt_dst', 'modbus_fn', 'modbus_response'
raison : missing some values

problème avec modbus_fn, lot of missing value (153121 for attack1), useless column ? Même chose avec modbus_response : Colonne lié au proto modbus et donc pas toute les colonne

La colonne proto contient les protocoles ARP ICMP Modbus TCP pour attack mais seulement Modbus et TCP pour le cas normal

Problème de typage : 'sport', 'dport', 'flags', 'n_pkt_src', 'n_pkt_dst'

Certaine dates sont partagé pour plusieurs entrées (ex : 2021-04-09 18:42:46.997219 a 4 entrés)

Le s dans ip_s, sport est pour source et le d pour destination

Il y a des cas noté normaux dans les csv attack, besoin du csv normal ?

different types d'attaques : 

 - ['normal' 'anomaly' 'MITM' 'physical fault'] pour attack_1
 - ['normal' 'physical fault' 'anomaly' 'DoS' 'MITM'] pour attack_3
 - ['normal' 'scan' 'DoS' 'physical fault' 'MITM'] pour attack_4 et attack_2

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'browser'

# KNN algorithm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
df_attack = pd.read_csv('dataset/Network datatset/cleaned/attack_4_cleaned.csv', delimiter=',', header=0)

In [4]:
df_attack.columns

Index(['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto',
       'flags', 'size', 'modbus_fn', 'modbus_response', 'n_pkt_src',
       'n_pkt_dst', 'label_n', 'label'],
      dtype='object')

In [5]:
df_attack.head(30)

,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,flags,size,modbus_fn,modbus_response,n_pkt_src,n_pkt_dst,label_n,label
0,2022-02-21 14:45:25.454111,74:46:a0:bd:a7:1b,e6:3f:ac:c9:a8:8c,84.3.251.20,84.3.251.101,60614,502,Modbus,11000,66,Read Holding Registers,-1,0,0,0,normal
1,2022-02-21 14:45:25.454114,74:46:a0:bd:a7:1b,fa:00:bc:90:d7:fa,84.3.251.20,84.3.251.103,60616,502,Modbus,11000,66,Read Holding Registers,-1,1,0,0,normal
2,2022-02-21 14:45:25.454142,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,60615,502,Modbus,11000,66,Read Holding Registers,-1,2,0,0,normal
3,2022-02-21 14:45:25.454260,00:80:f4:03:fb:12,74:46:a0:bd:a7:1b,84.3.251.18,84.3.251.20,502,60523,TCP,11000,109,NaN,-1,0,0,0,normal
4,2022-02-21 14:45:25.454365,e6:3f:ac:c9:a8:8c,74:46:a0:bd:a7:1b,84.3.251.101,84.3.251.20,502,60614,Modbus,11000,65,Read Holding Registers Response,0,0,1,0,normal
5,2022-02-21 14:45:25.454415,0a:fe:ec:47:74:fb,74:46:a0:bd:a7:1b,84.3.251.102,84.3.251.20,502,60615,Modbus,11000,65,Read Holding Registers Response,0,0,2,0,normal
6,2022-02-21 14:45:25.454416,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502,60616,Modbus,11000,65,Read Holding Registers Response,0,0,3,0,normal
7,2022-02-21 14:45:25.454720,00:80:f4:03:fb:12,74:46:a0:bd:a7:1b,84.3.251.18,84.3.251.20,502,60613,Modbus,11000,64,Read Coils Response,0,1,4,0,normal
8,2022-02-21 14:45:25.457141,74:46:a0:bd:a7:1b,e6:3f:ac:c9:a8:8c,84.3.251.20,84.3.251.101,60614,502,Modbus,11000,66,Read Holding Registers,-1,3,1,0,normal
9,2022-02-21 14:45:25.457143,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,60615,502,Modbus,11000,66,Read Holding Registers,-1,4,1,0,normal


In [21]:
X = df_attack[["size", "dport"]]
df_attack.loc[df_attack["label"] == "Normal", "label"] = 0
df_attack.loc[df_attack["label"] != "Normal", "label"] = 1


In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df_attack["label"])


In [6]:
print(df_attack[["size", "dport"]].dtypes)


size     int64
dport    int64
dtype: object


In [31]:
# Importer les bibliothèques nécessaires
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Sélectionner les caractéristiques (X) et la cible (y)
X = df_attack[["size", "dport"]]
y = df_attack["label"]  # Utilisation de df_attack["label"] pour avoir un vecteur 1D

X = X.fillna(0)
# Diviser le jeu de données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Créer le modèle KNN
k = 3  # Nombre de voisins à considérer
knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
# Ajuster le modèle
knn.fit(X_train, y_train)

# Faire des prédictions
y_pred = knn.predict(X_test)

# Évaluer les performances
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

# Afficher les résultats
print(f"Précision: {accuracy:.2f}")
print("Matrice de confusion:\n", conf_matrix)
print("Rapport de classification:\n", class_report)

# Optionnel : Visualiser les résultats
plt.scatter(X_test.iloc[:, 0], X_test.iloc[:, 1], c=y_pred, cmap='viridis', marker='o', edgecolor='k')
plt.title("Classification avec KNN")
plt.xlabel("Caractéristique 1")
plt.ylabel("Caractéristique 2")
plt.show()



# Pour chaque label, on compte le nombre de requêtes 
On regarde également les protocoles associés au label 

In [6]:
# Compter le nombre d'occurrences de chaque label en fonction de proto
df_counts = df_attack.groupby(['proto', 'label']).size().reset_index(name='count')

# Afficher les premières lignes du DataFrame
print(df_counts.head())


    proto           label    count
0  Modbus             DoS   333016
1  Modbus            MITM   443809
2  Modbus          normal  2703946
3  Modbus  physical fault   289827
4     TCP             DoS   399733


# Affichage 

In [7]:
def histogramme(df,x,y,color=None):

    # Créez le plot
    fig = px.histogram(df_counts, x=x,y=y,
                title='Nombre d\'attaques par protocole',
                labels={'proto': 'Protocole', 'label': 'Nombre d\'attaques'},
                color=color)

    # Affichez le graphique
    fig.show()


Histogramme montrant les différentes requêtes en fonction du protocole et en prenant en compte le label 

In [8]:
histogramme(df_counts,'proto',"count","label")

In [10]:
histogramme(df_counts,'label',"count")